# CTI calibration figures

Contact author: Alex Broughton
<br>Date: 


In [ ]:
! eups list -s | grep lsst_distrib

## Introduction

This notebook produces CTI figures for RTN-117:

- Serial EPER vs flat-field signal for one amplifier
- Global serial CTI histogram across the LSSTCam focal plane

Output PDFs are written to the current directory and copied into figures/ for the technote.


## 1.0 Set Up

In [ ]:
### Import packages and configure plotting defaults
from lsst.daf.butler import Butler
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

plt.rcParams.update({"font.size": 12})
import matplotlib as mpl
mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
})


butler = Butler("main")
camera = butler.get("camera", instrument="LSSTCam", collections="LSSTCam/defaults")

# Example amplifier used in the technote (detector 93 / R22-S10, amp C04)
DETECTOR = 93
AMP = "C04"


## 2.0 Serial EPER example

Plot measured serial EPER statistics for one amplifier, with PTC turn-off and roll-off markers and the fitted global CTI level.


In [ ]:
cti = butler.get("cti", instrument="LSSTCam", detector=DETECTOR, collections="LSSTCam/calib")
ptc = butler.get("ptc", instrument="LSSTCam", detector=DETECTOR, collections="LSSTCam/calib")

plt.figure()
plt.plot(
    cti.signals[AMP], cti.serialEper[AMP],
    "ko", markersize=5, markeredgecolor="none", alpha=0.15,
)
plt.axvline(ptc.ptcTurnoff[AMP] * ptc.gain[AMP], linestyle="--", color="k")
plt.axvline(ptc.ptcRolloff[AMP] * ptc.gain[AMP], linestyle="--", color="k")
plt.axvline(cti.serialCtiTurnoff[AMP], linestyle="--", color="k")
plt.axhline(cti.globalCti[AMP], linestyle="-", linewidth=2, color="cornflowerblue")

plt.text(ptc.ptcRolloff[AMP] * ptc.gain[AMP] - 5000, 0.05e-6, "PTC Roll-off", rotation=90)
plt.text(ptc.ptcTurnoff[AMP] * ptc.gain[AMP] - 5000, 0.05e-6, "PTC Turn-off", rotation=90)
plt.text(cti.serialCtiTurnoff[AMP] - 5000, 0.05e-6, "Serial CTI Turn-off", rotation=90)
plt.text(0.3e5, cti.globalCti[AMP] - 0.075e-6, "Global CTI", color="cornflowerblue")

plt.xlim(0, 1.45e5)
plt.ylim(0, 1.2e-6)
plt.ticklabel_format(style="sci", axis="x", scilimits=(0, 0), useMathText=True)
plt.ticklabel_format(style="sci", axis="y", scilimits=(0, 0), useMathText=True)
plt.xlabel(r"Flat field level, $\mu$ (electrons)")
plt.ylabel("Serial EPER")
plt.savefig("cti-det93-ampC04-serial-eper.pdf", format="pdf", bbox_inches="tight")


## 3.0 Global CTI histogram

Collect global CTI values from every CTI calibration dataset and plot the ITL and E2V distributions.


In [ ]:
refs = list(butler.query_datasets("cti", instrument="LSSTCam", collections="LSSTCam/calib"))

global_cti = []
det_type = []

for ref in tqdm(refs):
    detector = ref.dataId["detector"]
    pt = camera[detector].getPhysicalType()
    cti_ref = butler.get(ref)
    try:
        amps = [cti_ref.globalCti[amp.getName()] for amp in camera[detector]]
    except KeyError:
        continue
    global_cti.extend(amps)
    det_type.extend([pt] * len(amps))

global_cti = np.array(global_cti)
e2v = np.array(det_type) == "E2V"
itl = np.array(det_type) == "ITL"

plt.figure()
plt.hist(
    global_cti[itl], bins=100, histtype="step", label="ITL",
    color="darkorange", range=[0, 1e-5], linewidth=1.5, alpha=0.75,
)
plt.hist(
    global_cti[e2v], bins=100, histtype="step", label="E2V",
    color="cornflowerblue", range=[0, 1e-5], linewidth=1.5, alpha=0.75,
)
plt.xlim(0, 1e-5)
plt.ylim(0, 1e3)
plt.yscale("symlog", linthresh=25)
plt.xlabel("Global CTI (unitless)")
plt.ylabel("N(bin)")
plt.ticklabel_format(style="sci", axis="x", scilimits=(0, 0), useMathText=True)
plt.savefig("cti-det93-ampC04-global-cti-histogram.pdf", format="pdf", bbox_inches="tight")
